# Tutorial 5 - Batch Fallback, Timeouts, And Retry Queue

This tutorial demonstrates compatible grouping, 8-to-4-to-1 fallback, pending timeouts, and retry queue review.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import sqlite3
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

RUN_LIVE = os.getenv("BRAIN_SIM_RUN_LIVE") == "1"
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "examples").exists() else CWD.parent if CWD.name == "examples" else CWD
EXAMPLE_DIR = ROOT / "examples"
DATA_DIR = EXAMPLE_DIR / "data"
EXPECTED_DIR = EXAMPLE_DIR / "expected"
RUNS_DIR = EXAMPLE_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {ROOT}")
print(f"Live execution enabled: {RUN_LIVE}")


In [ ]:
@dataclass(frozen=True)
class TutorialRateLimitState:
    limit: int | None
    remaining: int | None
    reset_seconds: int | None


@dataclass(frozen=True)
class TutorialSubmitResult:
    status_code: int
    location: str
    body: Any
    headers: dict[str, str]
    rate_limit: TutorialRateLimitState


@dataclass(frozen=True)
class TutorialPollResult:
    status: str
    body: Any
    events: list[dict[str, Any]]


class CompleteFakeBrainClient:
    def __init__(self, alpha_prefix: str = "tutorial-alpha", location_prefix: str = "/simulations/tutorial") -> None:
        self.alpha_prefix = alpha_prefix
        self.location_prefix = location_prefix
        self.submit_count = 0
        self.locations: dict[str, list[str]] = {}

    def submit(self, payload: dict[str, Any] | list[dict[str, Any]]) -> TutorialSubmitResult:
        self.submit_count += 1
        payloads = payload if isinstance(payload, list) else [payload]
        alpha_ids = [f"{self.alpha_prefix}-{self.submit_count + offset}" for offset in range(len(payloads))]
        location = f"{self.location_prefix}-{self.submit_count}"
        self.locations[location] = alpha_ids
        return TutorialSubmitResult(
            status_code=201,
            location=location,
            body={"id": location.rsplit("/", 1)[-1]},
            headers={"Location": location, "X-Ratelimit-Remaining": "999"},
            rate_limit=TutorialRateLimitState(limit=None, remaining=999, reset_seconds=None),
        )

    def poll(self, location: str, *, timeout_seconds: float) -> TutorialPollResult:
        alpha_ids = self.locations[location]
        body: Any = {"alpha": alpha_ids[0]} if len(alpha_ids) == 1 else {"alphas": alpha_ids}
        return TutorialPollResult(status="complete", body=body, events=[{"body": body}])

    def fetch_alpha(self, alpha_id: str) -> dict[str, Any]:
        index = int(alpha_id.rsplit("-", 1)[-1])
        return {
            "id": alpha_id,
            "status": "UNSUBMITTED",
            "is": {
                "sharpe": f"{1.0 + index / 10:.2f}",
                "fitness": f"{0.5 + index / 10:.2f}",
                "returns": f"{0.03 + index / 100:.2f}",
                "turnover": f"{0.10 + index / 100:.2f}",
                "drawdown": f"{0.02 + index / 100:.2f}",
                "checks": [
                    {"name": "LOW_SHARPE", "result": "WARNING"} if index == 2 else {"name": "SELF_CORRELATION", "result": "PASS"}
                ],
            },
        }

    def fetch_recordset(self, alpha_id: str, recordset_name: str) -> dict[str, Any]:
        return {
            "alpha_id": alpha_id,
            "recordset": recordset_name,
            "rows": [
                {"date": "2026-01-01", "value": 0.01},
                {"date": "2026-01-02", "value": 0.02},
            ],
        }


In [ ]:
from brain_sim.excel import read_excel_expressions
from brain_sim.payloads import build_payload_record

excel_path = DATA_DIR / "tutorial_05_fallback_alphas.xlsx"
alpha_rows = read_excel_expressions(excel_path, sheet_name="alphas")
payload_records = []
for index, alpha in enumerate(alpha_rows, start=1):
    record = build_payload_record(alpha)
    payload_records.append(
        record.__class__(
            row_id=record.row_id,
            alpha_hash=f"fallback-hash-{index}",
            payload=record.payload,
            metadata=record.metadata,
        )
    )

print(f"Loaded {len(payload_records)} payload records from {excel_path.name}")
[(record.row_id, record.alpha_hash, record.payload["settings"]["universe"]) for record in payload_records]


## 1. Batch Compatibility

The runner batches only compatible payloads: type, instrument type, region, universe, delay, and language must match.


In [ ]:
from brain_sim.batch import iter_allowed_batch_chunks

print([len(chunk) for chunk in iter_allowed_batch_chunks(payload_records, 8)])


## 2. Rejected Multi-Submit Fallback And Timeout

This fake client rejects an 8-item request, accepts 4-item requests, completes the first 4 rows, and times out the second 4 rows.


In [ ]:
class FallbackTimeoutFakeClient(CompleteFakeBrainClient):
    def submit(self, payload: dict[str, Any] | list[dict[str, Any]]) -> TutorialSubmitResult:
        payloads = payload if isinstance(payload, list) else [payload]
        if len(payloads) > 4:
            return TutorialSubmitResult(
                status_code=422,
                location="",
                body={"error": "multi-submit payload too large"},
                headers={},
                rate_limit=TutorialRateLimitState(limit=None, remaining=None, reset_seconds=None),
            )
        self.submit_count += 1
        location = "/simulations/fallback-timeout" if self.submit_count == 2 else f"/simulations/fallback-{self.submit_count}"
        self.locations[location] = [f"tutorial-alpha-{index}" for index in range(1 + (self.submit_count - 1) * 4, 1 + self.submit_count * 4)]
        return TutorialSubmitResult(201, location, {"id": location}, {"Location": location}, TutorialRateLimitState(None, None, None))

    def poll(self, location: str, *, timeout_seconds: float) -> TutorialPollResult:
        if location == "/simulations/fallback-timeout":
            return TutorialPollResult(status="pending_timeout", body=None, events=[{"progress": 0.35}])
        return super().poll(location, timeout_seconds=timeout_seconds)


from brain_sim.batch import BatchRunner

run_dir = RUNS_DIR / "tutorial_05_fallback"
shutil.rmtree(run_dir, ignore_errors=True)
runner = BatchRunner(FallbackTimeoutFakeClient(), run_dir)
result = runner.run(payload_records, batch_size="auto", poll_timeout_seconds=1)
print(result)
print((run_dir / "retry_queue.jsonl").read_text(encoding="utf-8"))


## 3. Retry Review

Do not blindly resubmit ambiguous transport errors. Inspect `retry_queue.jsonl`, preserve `simulation_location`, and decide whether to wait longer, fix expressions, or retry only specific rows.
